# Rush E

A sample of the "Rush E" score, combining Codetto's `audio` and `scene3d` libraries. One sphere is created per pitch used in the piece, arranged left-to-right like a piano roll. As `audio.play_notes_async()` plays the piece in the background, `scene.on_frame` tracks elapsed time against the same note data and lights up each sphere the instant its note starts.

Use the "Run All Cells" at the top of this notebook and scroll down to the scene output.

In [ ]:
from codetto import scene3d, audio
import colorsys

scene = scene3d.Scene()
scene.set_sky(scene3d.Sky.PURE_SKY)
scene.ambient.set_brightness(55)
scene.camera.set_position(0, 5, -24).look_at(0, 1, 0)

In [ ]:
# One sphere per pitch used in the piece, arranged low-to-high like a piano roll,
# colored in a rainbow gradient across the row.
pitches = ['D#4', 'E4', 'F4', 'F#4', 'G#4', 'A4', 'B4', 'C5', 'D5', 'E5', 'F5', 'G#5', 'A5', 'B5', 'C6', 'D6']

REST_SCALE = 0.35
HIT_SCALE = 1.0
REST_COLOR = "#22222e"

spheres = {}
for i, pitch in enumerate(pitches):
  hue = i / len(pitches)
  r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 1.0)
  color = f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

  sphere = scene3d.Shapes.Sphere(diameter=1, segments=16)
  sphere.set_position(i * 1.3 - (len(pitches) - 1) * 1.3 / 2, 1, 0)
  sphere.set_color(REST_COLOR)
  sphere.set_scale(REST_SCALE, REST_SCALE, REST_SCALE)
  scene.add(sphere)

  spheres[pitch] = {"mesh": sphere, "color": color}

In [ ]:
#@title Note and Timing Sequence
sequence = [
  ("E4", 0.326), ("E4", 0.32), ("E4", 0.315), ("E4", 0.31), ("E4", 0.305), ("E4", 0.3),
  ("E4", 0.296), ("E4", 0.291), ("E4", 0.287), ("E4", 0.283), ("E4", 0.279), ("E4", 0.275),
  ("E4", 0.271), ("E4", 0.267), ("E4", 0.263), ("E4", 0.26), ("E4", 0.256), ("E4", 0.253),
  ("E4", 0.25), ("E4", 0.246), ("F4", 0.243), ("E4", 0.24), ("D#4", 0.237), ("E4", 0.466),
  ("A4", 0.455), ("C5", 0.879), ("D5", 0.214), ("D5", 0.211), ("D5", 0.209), ("D5", 0.207),
  ("D5", 0.204), ("C5", 0.202), ("B4", 0.2), ("D5", 0.198), ("C5", 0.196), ("C5", 0.194),
  ("C5", 0.192), ("C5", 0.19), ("C5", 0.188), ("B4", 0.187), ("A4", 0.185), ("C5", 0.183),
  ("B4", 0.181), ("B4", 0.18), ("B4", 0.178), ("B4", 0.176), ("F#4", 0.348), ("B4", 0.342),
  ("G#4", 1.309), ("E4", 0.162), ("E4", 0.164), ("E4", 0.161), ("E4", 0.159), ("E4", 0.157),
  ("E4", 0.155), ("E4", 0.153), ("E4", 0.151), ("E4", 0.149), ("E4", 0.148), ("E4", 0.146),
  ("E4", 0.144), ("E4", 0.142), ("E4", 0.141), ("E4", 0.139), ("E4", 0.137), ("E4", 0.136),
  ("F4", 0.134), ("E4", 0.133), ("D#4", 0.131), ("E4", 0.259), ("A4", 0.127), ("C5", 0.126),
  ("E5", 0.248), ("A5", 0.243), ("C6", 0.472), ("D6", 0.115), ("C6", 0.115), ("B5", 0.114),
  ("D6", 0.114), ("C6", 0.113), ("B5", 0.113), ("A5", 0.112), ("C6", 0.111), ("B5", 0.111),
  ("A5", 0.11), ("G#5", 0.11), ("B5", 0.109), ("A5", 0.109), ("E5", 0.108), ("C5", 0.108),
  ("A4", 0.107), ("F5", 0.107), ("E5", 0.107), ("D5", 0.106), ("C5", 0.106), ("B4", 0.105),
  ("A4", 0.105), ("G#4", 0.104), ("B4", 0.104), ("A4", 0.496),
]

In [ ]:
# Precompute when each note starts, so on_frame can look up "what's playing now"
cues = []
t = 0.0
for note, duration in sequence:
  cues.append((t, note))
  t += duration

elapsed = 0.0
cue_index = 0
active_pitch = None

def light(pitch, hit):
  sphere = spheres[pitch]
  scale = HIT_SCALE if hit else REST_SCALE
  sphere["mesh"].set_scale(scale, scale, scale)
  sphere["mesh"].set_color(sphere["color"] if hit else REST_COLOR)

@scene.on_frame
def animate(dt):
  global elapsed, cue_index, active_pitch
  elapsed += dt
  while cue_index < len(cues) and elapsed >= cues[cue_index][0]:
    if active_pitch is not None:
      light(active_pitch, False)
    active_pitch = cues[cue_index][1]
    light(active_pitch, True)
    cue_index += 1

await audio.play_notes_async(sequence)
scene.run()